In [ ]:
%%configure  

{ 
    "vCores": 
    { 
        "parameterName": "pipelinecore", 
        "defaultValue": 4 
    }
}

In [ ]:
!pip install -q duckrun --upgrade
notebookutils.session.restartPython()


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
try:
    import notebookutils
    vl             = notebookutils.variableLibrary.getLibrary("deploy_config")
    workspace_id   = vl.workspace_id
    lakehouse_name = vl.lakehouse_name
    download_limit = vl.download_limit
    process_limit  = vl.process_limit
    lakehouse_id   = notebookutils.lakehouse.get(lakehouse_name).get('id')
    token          = notebookutils.credentials.getToken('storage')
    dbt_target     = 'dev'
    notebookutils.fs.cp(
        f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/dbt',
        '/tmp',
        True,
    )
    dbt_path = '/tmp/dbt'
except ModuleNotFoundError:
    from azure.identity import AzureCliCredential
    import yaml
    from pathlib import Path
    _root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "deploy_config.yml").exists()), None)
    if _root is None:
        raise FileNotFoundError("deploy_config.yml not found in cwd or any parent — run from inside the cloned repo")
    _all   = yaml.safe_load((_root / "deploy_config.yml").read_text())
    _cfg   = {**_all.get("defaults", {}), **_all["local"]}
    workspace_id   = _cfg["ws"]
    lakehouse_id   = _cfg["lakehouse"]
    lakehouse_name = _cfg["lakehouse_name"]
    download_limit = _cfg["download_limit"]
    process_limit  = _cfg["process_limit"]
    dbt_path       = _cfg["dbt_path"]
    token          = AzureCliCredential().get_token("https://storage.azure.com/.default").token
    dbt_target     = 'dev'
os.environ['download_limit']   = download_limit
os.environ['process_limit']    = process_limit

In [ ]:
os.environ['FILES_PATH']          = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files'
os.environ['ONELAKE_TABLES_PATH'] = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables'
os.environ['ONELAKE_TOKEN']       = token

: 

In [ ]:
from dbt.cli.main import dbtRunner
os.chdir(dbt_path)
dbt = dbtRunner()
result = dbt.invoke(["run", "--target", dbt_target, "--profiles-dir", "."])
if not result.success:
    print("dbt run had failures — retrying failed models once...")
    _ = dbt.invoke(["retry", "--target", dbt_target, "--profiles-dir", "."])
_ = dbt.invoke(["test", "--target", dbt_target, "--profiles-dir", "."])

12:00:05  Running with dbt=1.11.8
12:00:08  Registered adapter: duckrun=0.3.1
12:00:21  [WARNING][MissingArgumentsPropertyInGenericTestDeprecation]: Deprecated
functionality
Found top-level arguments to test `accepted_values` defined on 'fct_price_today'
in package 'aemo_electricity' (models\marts\schema.yml). Arguments to generic
tests should be nested under the `arguments` property.
12:00:23  Found 8 models, 35 data tests, 1 operation, 484 macros
12:00:23  
12:00:23  Concurrency: 1 threads (target='dev')
12:00:23  
12:00:45  1 of 1 START hook: aemo_electricity.on-run-start.0 ............................. [RUN]
12:00:45  1 of 1 OK hook: aemo_electricity.on-run-start.0 ................................ [OK in 0.03s]
12:00:45  
12:00:45  1 of 8 START sql table model mart.dim_calendar ................................. [RUN]
12:01:20  Unhandled error while executing 
Generic MicrosoftAzure error
          ↳ Error performing bulk delete request
           ↳ Error performing POST https://one